<h5> Imports and load

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

pd.set_option('display.max_columns', None)

df = pd.read_csv('dataset.csv')
target_col = 'default payment next month'

print(f"Shape: {df.shape}")

Shape: (24028, 27)


<h11> SimpleImputer from sklearn is the tool that enforces the "fit on train only" rule cleanly -> it has a .fit() step (learns the mean/mode from whatever data you give it) and a separate .transform() step (applies that learned value elsewhere), which naturally prevents you from accidentally mixing train and test statistics.

<h5> Drop unnecessary columns

In [8]:
df = df.drop(columns=['ID', 'WORK_YEARS'])
print(f"Shape after dropping ID and WORK_YEARS: {df.shape}")
df.columns.tolist()

Shape after dropping ID and WORK_YEARS: (24028, 25)


['LIMIT_BAL',
 'JOB_TYPE',
 'SEX',
 'EDUCATION',
 'MARRIAGE',
 'AGE',
 'PAY_0',
 'PAY_2',
 'PAY_3',
 'PAY_4',
 'PAY_5',
 'PAY_6',
 'BILL_AMT1',
 'BILL_AMT2',
 'BILL_AMT3',
 'BILL_AMT4',
 'BILL_AMT5',
 'BILL_AMT6',
 'PAY_AMT1',
 'PAY_AMT2',
 'PAY_AMT3',
 'PAY_AMT4',
 'PAY_AMT5',
 'PAY_AMT6',
 'default payment next month']

<h11> ID carries no predictive information (just a row identifier), and WORK_YEARS is 85% missing - both decided back in Notebook 1. Doing this before the split is safe since it doesn't involve any computed statistic (mean/mode) that could leak.

<h5> Train/test split

In [9]:
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"\nTrain target distribution:\n{y_train.value_counts(normalize=True)}")
print(f"\nTest target distribution:\n{y_test.value_counts(normalize=True)}")

Train shape: (16819, 24), Test shape: (7209, 24)

Train target distribution:
default payment next month
0    0.972353
1    0.027647
Name: proportion, dtype: float64

Test target distribution:
default payment next month
0    0.972396
1    0.027604
Name: proportion, dtype: float64


<h11> Split happens before any imputation, per our decision. test_size=0.3 matches the reference project's 70:30 ratio. stratify=y is important given the severe imbalance — it ensures both train and test sets preserve the same ~2.76%/97.24% class ratio; without it, random splitting could accidentally concentrate more (or fewer) of the rare defaulters into one set. random_state=42 makes the split reproducible — you'll get the exact same split every time you rerun this, which matters for consistent results across your baseline/undersampling/oversampling notebooks later.

<h5> Categorical missing values → "UNKNOWN"

In [10]:
categorical_cols = ['SEX', 'EDUCATION', 'MARRIAGE', 'JOB_TYPE']

for col in categorical_cols:
    X_train[col] = X_train[col].fillna('UNKNOWN')
    X_test[col] = X_test[col].fillna('UNKNOWN')

print("Missing values remaining in categoricals:")
print(X_train[categorical_cols].isnull().sum())

Missing values remaining in categoricals:
SEX          0
EDUCATION    0
MARRIAGE     0
JOB_TYPE     0
dtype: int64


<h11> "UNKNOWN" is a constant, not a computed statistic — so unlike mean imputation, there's no leakage risk here even applying it directly to both sets without a "fit" step. Note JOB_TYPE is included here now (unlike the reference project, which filled it with numeric 16) — this is consistent with our decision to treat it as a nominal categorical throughout, missing values included.

<h5> PAY_0–PAY_6 missing values → 0 (mode)

In [11]:
pay_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

for col in pay_cols:
    X_train[col] = X_train[col].fillna(0)
    X_test[col] = X_test[col].fillna(0)

print("Missing values remaining in PAY columns:")
print(X_train[pay_cols].isnull().sum())

Missing values remaining in PAY columns:
PAY_0    0
PAY_2    0
PAY_3    0
PAY_4    0
PAY_5    0
PAY_6    0
dtype: int64


<h11> 0 is our chosen constant (the mode, representing "no delinquency") rather than the reference project's invented 9. Same reasoning as above — a fixed constant doesn't require fitting on train only, since it's not derived from the data's statistics at all, just a domain-informed choice we made in advance.

<h5> Remaining numerics → mean imputation (fit on train only)

In [12]:
from sklearn.impute import SimpleImputer

numeric_impute_cols = ['LIMIT_BAL', 'AGE',
                        'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
                        'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

imputer = SimpleImputer(strategy='mean')
imputer.fit(X_train[numeric_impute_cols])   # fit on TRAIN ONLY

X_train[numeric_impute_cols] = imputer.transform(X_train[numeric_impute_cols])
X_test[numeric_impute_cols] = imputer.transform(X_test[numeric_impute_cols])

print("Missing values remaining in numeric columns:")
print(X_train[numeric_impute_cols].isnull().sum())
print(X_test[numeric_impute_cols].isnull().sum())

Missing values remaining in numeric columns:
LIMIT_BAL    0
AGE          0
BILL_AMT1    0
BILL_AMT2    0
BILL_AMT3    0
BILL_AMT4    0
BILL_AMT5    0
BILL_AMT6    0
PAY_AMT1     0
PAY_AMT2     0
PAY_AMT3     0
PAY_AMT4     0
PAY_AMT5     0
PAY_AMT6     0
dtype: int64
LIMIT_BAL    0
AGE          0
BILL_AMT1    0
BILL_AMT2    0
BILL_AMT3    0
BILL_AMT4    0
BILL_AMT5    0
BILL_AMT6    0
PAY_AMT1     0
PAY_AMT2     0
PAY_AMT3     0
PAY_AMT4     0
PAY_AMT5     0
PAY_AMT6     0
dtype: int64


<h11> This is the one place leakage was a genuine risk, and SimpleImputer.fit() is called only on X_train — exactly the pattern we just discussed. Both .isnull().sum() checks confirm zero missing values remain in both sets after this step.

<h5> Full missing value check across everything

In [13]:
print("Train set - any remaining missing values?")
print(X_train.isnull().sum().sum())

print("\nTest set - any remaining missing values?")
print(X_test.isnull().sum().sum())

Train set - any remaining missing values?
0

Test set - any remaining missing values?
0


<h11> A final sanity check summing across all columns at once — should print 0 for both if every column has been handled. This is the checkpoint before we move to encoding.

<h5>  One-hot encoding (with train/test alignment)

In [14]:
categorical_cols_encode = ['SEX', 'EDUCATION', 'MARRIAGE', 'JOB_TYPE']

X_train_encoded = pd.get_dummies(X_train, columns=categorical_cols_encode)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols_encode)

print(f"X_train_encoded shape: {X_train_encoded.shape}")
print(f"X_test_encoded shape: {X_test_encoded.shape}")

train_cols = set(X_train_encoded.columns)
test_cols = set(X_test_encoded.columns)

print(f"\nColumns in train but not test: {train_cols - test_cols}")
print(f"Columns in test but not train: {test_cols - train_cols}")

X_train_encoded shape: (16819, 50)
X_test_encoded shape: (7209, 50)

Columns in train but not test: set()
Columns in test but not train: set()


<h11> This is the exact risk I flagged — pd.get_dummies() only creates columns for categories it actually sees in the data you give it. If, say, EDUCATION_post graduate happens to appear in your training set but by chance no test-set customer has that value (or vice versa), the two encoded dataframes end up with a different number of columns entirely — and any model trained on one set literally cannot be applied to the other, since the column structure won't match. This check tells us immediately whether that's happened.

Both shapes match (50 columns each) and both mismatch sets are empty — so no misalignment happened this time, train and test saw the exact same set of categories for every categorical column.

<h5> Fix any column mismatch (align train/test)

In [15]:
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='outer', axis=1, fill_value=0)

print(f"X_train_encoded shape after align: {X_train_encoded.shape}")
print(f"X_test_encoded shape after align: {X_test_encoded.shape}")
print(f"\nColumns match: {list(X_train_encoded.columns) == list(X_test_encoded.columns)}")

X_train_encoded shape after align: (16819, 50)
X_test_encoded shape after align: (7209, 50)

Columns match: True


<h11> .align() is pandas' built-in tool for exactly this problem — it takes the union of all columns from both dataframes, and fills in 0 for any column that's missing from one side (e.g., if a test customer never had EDUCATION_post graduate, that column gets added to the test set filled entirely with 0s, since 0 correctly represents "not this category" for a one-hot column). Even if Block 8 shows no mismatch this time, it's good practice to always run this step — it costs nothing when there's no mismatch, and protects you automatically if it ever does happen (e.g., if you change your train/test split or resampling strategy later).

<h5> Final check and save cleaned datasets

In [16]:
print("Final X_train_encoded info:")
X_train_encoded.info()

print("\nSample of encoded columns:")
print(X_train_encoded.columns.tolist())

Final X_train_encoded info:
<class 'pandas.DataFrame'>
Index: 16819 entries, 10634 to 7161
Data columns (total 50 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   LIMIT_BAL                      16819 non-null  float64
 1   AGE                            16819 non-null  float64
 2   PAY_0                          16819 non-null  float64
 3   PAY_2                          16819 non-null  float64
 4   PAY_3                          16819 non-null  float64
 5   PAY_4                          16819 non-null  float64
 6   PAY_5                          16819 non-null  float64
 7   PAY_6                          16819 non-null  float64
 8   BILL_AMT1                      16819 non-null  float64
 9   BILL_AMT2                      16819 non-null  float64
 10  BILL_AMT3                      16819 non-null  float64
 11  BILL_AMT4                      16819 non-null  float64
 12  BILL_AMT5                      

<h5> Save block to CSVs

In [17]:
X_train_encoded.to_csv('X_train_clean.csv', index=False)
X_test_encoded.to_csv('X_test_clean.csv', index=False)
y_train.to_csv('y_train_clean.csv', index=False)
y_test.to_csv('y_test_clean.csv', index=False)

print("Saved: X_train_clean.csv, X_test_clean.csv, y_train_clean.csv, y_test_clean.csv")

Saved: X_train_clean.csv, X_test_clean.csv, y_train_clean.csv, y_test_clean.csv


<h11>
Quick recap of what we've built and locked in:

- Dropped ID, WORK_YEARS
- Split first (70:30, stratified) — before any statistic-based imputation, avoiding the leakage issue present in the reference project
- Categorical + JOB_TYPE missing values → "UNKNOWN"
- PAY_0–PAY_6 missing values → 0 (mode), instead of the reference project's invented 9
- Remaining numerics → mean imputation, fit on train only
- JOB_TYPE one-hot encoded (unlike the reference project, which left it raw)
- Train/test column alignment verified
- Saved as X_train_clean.csv, X_test_clean.csv, y_train_clean.csv, y_test_clean.csv